In [1]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT = Path('/kaggle/input/competitions/ml-gen-x-bioreasoning-challenge-track-a')
LOCAL_DATA = PROJECT_ROOT / 'data'

DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_DATA
print(f'Using data directory: {DATA_DIR}')

if Path('/kaggle/input').exists():
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))


/kaggle/input/competitions/ml-gen-x-bioreasoning-challenge-track-a/train.csv
/kaggle/input/competitions/ml-gen-x-bioreasoning-challenge-track-a/test.csv
/kaggle/input/datasets/nident/prompts/prompt.txt


In [2]:
SEEDS = [42, 43, 44]

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

prompt_candidates = [
    Path('/kaggle/input/datasets/nident/prompts/prompt.txt'),
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '04_combined_de_first_fewshot_strict.txt',
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '00_zero_shot_strict.txt',
]
default_prompt_path = next((path for path in prompt_candidates if path.exists()), None)
if default_prompt_path is None:
    raise FileNotFoundError('No prompt template found in Kaggle input or local prompt_strategies/track_a')

default_prompt = default_prompt_path.read_text(encoding='utf-8')

print(f'Using prompt template: {default_prompt_path}')
print(default_prompt[:1000])
print(train.columns.tolist())
print(train.head(2))
print(test.head(2))


# Track A -- default mlgenx ternary zero-shot prompt

You are an expert molecular biologist who studies how genes are related using Perturb-seq.

Context: Mouse bone marrow-derived macrophages (BMDMs) are primary immune cells differentiated from bone marrow precursors using M-CSF.

Question: If you knockdown {pert} using CRISPRi in mouse BMDMs, what is the effect on {gene}?

Your answer must be one of:
A) Knockdown of {pert} results in up-regulation of {gene}.
B) Knockdown of {pert} results in down-regulation of {gene}.
C) Knockdown of {pert} does not significantly affect {gene}.

Answer:
Index(['id', 'pert', 'gene', 'label'], dtype='object')
            id   pert   gene label
0  Psmd4_Anxa2  Psmd4  Anxa2  down
1    Cul2_Upp1   Cul2   Upp1  none
                     id     pert            gene
0         Slc35b1_Pdia6  Slc35b1           Pdia6
1  Rprd2_9930111J21Rik2    Rprd2  9930111J21Rik2


In [ ]:
import os
from openai import OpenAI

api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print('OPENROUTER_API_KEY is not set; skipping live OpenRouter smoke test.')
else:
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
    )

    sample = test.iloc[0]
    prompt = default_prompt.format(
        pert=sample['pert'],
        gene=sample['gene'],
        cell_desc='Mouse bone marrow-derived macrophages (BMDMs) are primary immune cells differentiated from bone marrow precursors using M-CSF.',
    )

    response = client.chat.completions.create(
        model='openai/gpt-oss-120b:free',
        messages=[{'role': 'user', 'content': prompt}],
        extra_body={'reasoning': {'enabled': True}},
    )

    print(response.choices[0].message.content)
